In [5]:
import pandas as pd 

In [6]:
import pandas as pd

df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")

# Remove index column
df.drop(columns=['Unnamed: 0'], inplace=True)

# Fill missing text
df['Title'] = df['Title'].fillna("No Title")
df['Review Text'] = df['Review Text'].fillna("No Review")

# Fill categorical values
for col in ['Division Name','Department Name','Class Name']:
    df[col] = df[col].fillna("Unknown")

# Feature Engineering
df['Review_Length'] = df['Review Text'].apply(lambda x: len(x.split()))

df['Age_Group'] = pd.cut(df['Age'],
                         bins=[0,20,30,40,50,60,100],
                         labels=['Teen','20s','30s','40s','50s','60+'])

df['Is_Positive_Rating'] = df['Rating'].apply(lambda x: 1 if x >= 4 else 0)

In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# load data
df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")

df = df.drop(columns=['Unnamed: 0'])

df['Review Text'] = df['Review Text'].fillna("")
df['Title'] = df['Title'].fillna("")

# feature engineering
df['text'] = df['Title'] + " " + df['Review Text']

X = df[['text','Age','Rating','Positive Feedback Count']]
y = df['Recommended IND']

# preprocessing
preprocess = ColumnTransformer([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000), 'text'),
    ('num', 'passthrough', ['Age','Rating','Positive Feedback Count'])
])

# models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=120),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier()
}

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

for name, model in models.items():

    pipe = Pipeline([
        ('prep', preprocess),
        ('model', model)
    ])

    pipe.fit(X_train,y_train)

    pred = pipe.predict(X_test)

    acc = accuracy_score(y_test,pred)

    print(name,"Accuracy:",acc)

Logistic Regression Accuracy: 0.9406130268199234
Naive Bayes Accuracy: 0.8465304384844615
Linear SVM Accuracy: 0.9355044699872286
Random Forest Accuracy: 0.9382716049382716
Gradient Boosting Accuracy: 0.9320987654320988
KNN Accuracy: 0.922520221370796


In [8]:
import pickle

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# train final model (best model)
pipe = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)

# save model
pickle.dump(pipe, open("model.pkl", "wb"))

print("Model saved successfully!")

Model saved successfully!
